## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Keras-Tensorflow. Uso de Functional API
*****

## Librerias

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from time import time
from numpy import argmin
from pandas import read_excel
import matplotlib.pyplot as plt
%matplotlib inline

## Pre-processing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Keras from tensorflow
from tensorflow.keras.models import Model
from tensorflow.keras import layers

### Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model
  
  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics 
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Dataset

<center>
    <img src=https://blog.certifiedmtp.com/wp-content/uploads/2024/07/ASTM-C39-Mastering-Compressive-Strength-Tests-on-Concrete.jpg width=800>
</center>

El hormigón es el material más importante en ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de la edad y los ingredientes.

El conjunto de datos contiene información de los ingredientes de las mezclas con 1030 muestras. La descripción es la siguiente:

__Descripcion__:
<center>

| **Name**             | **Data Type** | **Measurement**    | **Description** |
|----------------------|---------------|--------------------|-----------------|
| Cement | Quantitative  | kg in a m3 mixture | Input Variable  |
| Blast Furnace Slag | Quantitative | kg in a m3 mixture | Input Variable |
| Fly Ash | Quantitative | kg in a m3 mixture | Input Variable |
| Water | Quantitative | kg in a m3 mixture | Input Variable |
| Superplasticizer | Quantitative | kg in a m3 mixture | Input Variable |
| Coarse Aggregate | Quantitative | kg in a m3 mixture | Input Variable |
| Fine Aggregate | Quantitative | kg in a m3 mixture | Input Variable |
| Age | Quantitative | Day (1~365) | Input Variable |
| Concrete compressive strength | Quantitative | MPa | Output Variable |

</center>

**Objetivo**: Predecir Concrete compressive strength (Strength)

* La data y detalles está completamente disponible en [UCI Repository: Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength)

#### Carga de datos

In [ ]:
## Load data
data = read_excel('https://github.com/adoc-box/Datasets/raw/refs/heads/main/Concrete_Data.xls')
data.head(4)

In [ ]:
data.describe()

## Preprocesamiento de datos

In [ ]:
## Predictors and target assignment
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

## Partition sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=84)

## scaling
scale = StandardScaler().fit(X_train)
X_train = scale.transform(X_train)
X_test = scale.transform(X_test)

## Display data shape
print('(train shape) X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('(test shape) X: {}, y: {}'.format(X_test.shape, y_test.shape))

### Diseño del modelo

In [ ]:
## Network flow
Input = layers.Input(shape=(X_train.shape[1],), name='Input')
x = layers.Dense(units=12, activation='relu', name='Dense_1')(Input)
output = layers.Dense(units=1, activation='linear', name='output')(x)

## Model Instance
model = Model(inputs=Input, outputs=output, name='Regression')

## Compiler setting
model.compile(optimizer='sgd', 
              loss='mae', 
              metrics=['mse'])

model.summary()

In [ ]:
start = time()

## model fitting
history = model.fit(x=X_train, y=y_train, 
                    validation_split=0.15,
                    epochs=200, 
                    batch_size=16)

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
plot_history(history, width=14)

## Searching for the best epoch
id_min = argmin(history.history['val_loss'])
print('MAE - Validation: {} - Error: {}'.format(id_min+1, history.history['val_loss'][id_min]))

id_min = argmin(history.history['val_mse'])
print('MSE - Validation: {} - Error: {}'.format(id_min+1, history.history['val_mse'][id_min]))

## Mejor modelo

In [ ]:
## Network flow
Input = layers.Input(shape=(X_train.shape[1],), name='Input')
x = layers.Dense(units=12, activation='relu', name='Dense_1')(Input)
output = layers.Dense(units=1, activation='linear', name='output')(x)

## Model Instance
model = Model(inputs=Input, outputs=output, name='Regression')

## Compiler setting
model.compile(optimizer='sgd', 
              loss='mae', 
              metrics=['mse'])

In [ ]:
start = time()

## model fitting
history = model.fit(x=X_train, y=y_train, 
                    epochs=175, 
                    batch_size=16)

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
## Model evaluate
model.evaluate(X_test, y_test)

In [ ]:
## Compute prediction
prediction = model.predict(X_test)
prediction

In [ ]:
## Display results
plt.figure(figsize=(18, 5))
plt.plot(y_test.values, label='Ground Truth')
plt.plot(prediction, label='Prediction')
plt.ylabel(data.columns[-1])
plt.xlabel('sample')
plt.title('Ground truth vs prediction')
plt.legend()
plt.tight_layout()
plt.show()